In [2]:
# importing used libraries
import pandas as pd
import numpy as np
import re

In [4]:
# importing the uncleaned data
df=pd.read_csv('startup_funding.csv')
df.head()

,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


# Problems in data



- drop `remarks` column due to a large number of missing values.
- remove rows containing missing values from all columns except `subvertical`, since the percentage of missing values is relatively low.

- `amount`
    - remove special characters such as `,` and `+`.
    - standardize all funding amounts to Indian Rupees.
    - convert funding values to Crores for easier analysis.
    - convert the column from string to numeric datatype.

- `date`
    - correct malformed date entries such as `01/012015` and `01/01/015`.
    - standardize date formats across the dataset.
    - convert the column to datetime format.

- `startup`
    - remove website extensions such as `.com` and `.io`.
    - fix improperly encoded characters such as `\\xe2\\x80\\x99` and `\\xc2\\xa0`.
    - standardize startup names by resolving case inconsistencies (e.g., `Byju's` and `byju's`).

- `vertical`
    - fix improperly encoded characters such as `\\xc2\\xa0`.
    - standardize category names where required.

- `city`
    - standardize city names to remove duplicate representations (e.g., `Delhi` → `New Delhi`, `Gurgaon` → `Gurugram`).
    - fix improperly encoded characters such as `\\xc2\\xa0`.

- `investors`
    - convert all investor names to lowercase to eliminate duplicates caused by inconsistent capitalization.
    - standardize values such as `Undisclosed`, `Undisclosed Investor`, etc.
    - fix improperly encoded characters such as `\\xe2\\x80\\x99` and `\\xc2\\xa0`.


- set `Sr No` as the dataframe index.
- rename columns to shorter and more analysis-friendly names.

In [5]:
df['Investors Name']=df['Investors Name'].fillna('undisclosed')

In [6]:
df.drop(columns=['Remarks'],inplace=True)  # droping remarks columns due to high no.of missing values
df.set_index('Sr No',inplace=True)   # setting Sr No as index

In [7]:
# renaming column for comfortable analysis 
df.rename(columns={'Date dd/mm/yyyy':'date',
                  'Startup Name':'startup',
                  'Industry Vertical':'vertical',
                   'SubVertical':'subvertical',
                   'City  Location':'city',
                   'Investors Name':'investors',
                   'InvestmentnType':'round',
                   'Amount in USD':'amount'
                  },inplace=True)

In [8]:
df.head()

,date,startup,vertical,subvertical,city,investors,round,amount
Sr No,,,,,,,,
1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000"
2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394"
3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860"
4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000"
5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000"


In [9]:
df[df['amount'].isnull()]

,date,startup,vertical,subvertical,city,investors,round,amount
Sr No,,,,,,,,
145,01/10/2018,Northmist,Fashion,Mens Wear,Delhi,Prashant Jaiswal,Seed/ Angel Funding,NaN
156,04/09/2018,HappyGoEasy,Consumer Internet,Online Travel Agecy,Gurugram,"Korea Investment Partners (KIP), Samsung and C...",Private Equity,NaN
158,05/09/2018,Mad Street Den,Technology,Computer Vision And Artificial Intelligence (A...,Chennai,KDDI\\xc2\\xa0,Private Equity,NaN
166,01/08/2018,HealthFin,Finance,Patient Financing Platform,Pune,"Axilor, Sprout Venture Partners and others",Seed/ Angel Funding,NaN
190,01/07/2018,Leena AI,Technology,HR Virtual Agent For Employees,Gurugram,Y Combinator,Seed/ Angel Funding,NaN
...,...,...,...,...,...,...,...,...
3028,21/05/2015,Knit,NaN,NaN,NaN,"Rohit Jain, Amit Rambhia & Others",Seed Funding,NaN
3031,22/01/2015,Freshmonk,NaN,NaN,NaN,"August Capital Partners, Michael Blakey",Seed Funding,NaN
3032,22/01/2015,Englishleap.com,NaN,NaN,NaN,ANALEC,Private Equity,NaN


In [10]:
df['amount']=df['amount'].fillna('0')   # filling null values
df['amount']=df['amount'].str.replace(r'[^\d]+','',regex=True)  # extracting only digits using regex
df['amount']=df['amount'].replace('','0')  # replace '' to '0'
df['amount']=df['amount'].astype('float')  # typecasting to float

In [11]:
df['amount']=df['amount'].apply(lambda x: (x*92)/10000000)   # coverting to ind ruppes (crores)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3044 entries, 1 to 3044
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         3044 non-null   object 
 1   startup      3044 non-null   object 
 2   vertical     2873 non-null   object 
 3   subvertical  2108 non-null   object 
 4   city         2864 non-null   object 
 5   investors    3044 non-null   object 
 6   round        3040 non-null   object 
 7   amount       3044 non-null   float64
dtypes: float64(1), object(7)
memory usage: 214.0+ KB


In [13]:
# fixing dates like 01/01/015 and 01/012015
def date_fixer(text):
    x=re.sub(r'[^\d]','',text)   
    return re.sub(r'(\d{2})(\d{2})\d+',r'\1/\2/',x)+'20'   

In [14]:
df['date']=df['date'].apply(lambda x: date_fixer(x)+x[-2:])

In [15]:
df['date']=df['date'].astype('datetime64[ns]')  # typecasting to datetime format

In [16]:
df.dropna(subset=['date', 'startup', 'vertical', 'city', 'investors',
       'round', 'amount'],inplace=True)   # droping missing values based on all columns except 'subvertical' column

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2860 entries, 1 to 2873
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         2860 non-null   datetime64[ns]
 1   startup      2860 non-null   object        
 2   vertical     2860 non-null   object        
 3   subvertical  2098 non-null   object        
 4   city         2860 non-null   object        
 5   investors    2860 non-null   object        
 6   round        2860 non-null   object        
 7   amount       2860 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 201.1+ KB


In [18]:
df['startup']=df['startup'].apply(lambda x: x.lower())  # getting all startups to lowercase to remove duplicate startups like Byju's and byju's
df['investors']=df['investors'].apply(lambda x: x.lower())  # # getting all investors to lowercase to remove duplicate investors like IDG Ventures and idg ventures

In [19]:
# removing extensions like '.com' and '.io'
df['startup']=df['startup'].replace(r'\.com','',regex=True) 
df['startup']=df['startup'].replace(r'\.io','',regex=True)
# fixed parts which could not be covert to utf-8
df['startup']=df['startup'].replace(r'\\\\xe2\\\\x80\\\\x99',"'",regex=True)
df['startup']=df['startup'].replace(r'\\\\xc2\\\\xa0','',regex=True)

In [20]:
# fixed parts which could not be covert to utf-8
df['vertical']=df['vertical'].replace(r'\\\\xc2\\\\xa0','',regex=True)

In [21]:
# fixed parts which could not be covert to utf-8
df['investors']=df['investors'].replace(r'\\\\xe2\\\\x80\\\\x99',"'",regex=True)
df['investors']=df['investors'].replace(r'\\\\xc2\\\\xa0','',regex=True)

In [22]:
# renaming some cities 
df['city'].replace({'Delhi':'New Delhi',
                  'Bangalore':'Bengaluru',
                  'Gurgaon':'Gurugram'},inplace=True
                  )
# fixed parts which could not be covert to utf-8
df['city']=df['city'].replace(r'\\\\xc2\\\\xa0','',regex=True)

C:\Users\shilank\AppData\Local\Temp\ipykernel_12672\4014570949.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['city'].replace({'Delhi':'New Delhi',


In [23]:
df

,date,startup,vertical,subvertical,city,investors,round,amount
Sr No,,,,,,,,
1,2020-09-01,byju’s,E-Tech,E-learning,Bengaluru,tiger global management,Private Equity Round,1840.000000
2,2020-01-13,shuttl,Transportation,App based shuttle service,Gurugram,susquehanna growth equity,Series C,74.045225
3,2020-09-01,mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,sequoia capital india,Series B,168.901512
4,2020-02-01,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,vinod khatumal,Pre-series A,27.600000
5,2020-02-01,fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,sprout venture partners,Seed Round,16.560000
...,...,...,...,...,...,...,...,...
2869,2015-04-29,tracxn,Startup Analytics platform,NaN,Bengaluru,saif partners,Private Equity,32.200000
2870,2015-04-29,dazo,Mobile Food Ordering app,NaN,Bengaluru,"sumit jain, aprameya radhakrishna, alok goel, ...",Seed Funding,0.000000
2871,2015-04-29,tradelab,Financial Markets Software,NaN,Bengaluru,rainmatter,Seed Funding,3.680000


In [25]:
df.to_csv('cleaned_startup_data.csv',index=False)